In [ ]:
# ran the code in VSCode
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon


BASE = Path("/Users/yiyuanemmazhou/Documents/New project")

OBS_CRASH = BASE / "boston_daily_crashes_by_intersection_2018_2026.csv"
COORD_SRC = BASE / "bostoncrashsproad20182026.csv"
PRED_CRASH = BASE / "forecast_daily_intersection_2026_2030.csv"
WEATHER = BASE / "weatherdataboston.csv"
WIND_DIR = BASE / "boston_wind_direction_2018-01-01_to_2026-02-07_merged.csv"
BASEMAP = BASE / "boston_neighborhoods.geojson"

OUT_2020_2025 = BASE / "aggregate_anisotropic_heatmap_2020_2025.png"
OUT_2026_2028 = BASE / "aggregate_anisotropic_heatmap_2026_2028.png"

# KDE shape tuning
H_DEG = 0.0032
C_SCALE = 0.03
GRID_N = 280
EPS = 1e-12


def interp_linear(series: pd.Series) -> pd.Series:
    return series.interpolate(method="linear", limit_direction="both")


def load_coords() -> pd.DataFrame:
    df = pd.read_csv(COORD_SRC, usecols=["intersection", "latitude", "longitude"])
    df["latitude"] = pd.to_numeric(df["latitude"], errors="coerce")
    df["longitude"] = pd.to_numeric(df["longitude"], errors="coerce")
    df = df.dropna(subset=["intersection", "latitude", "longitude"])
    return (
        df.groupby("intersection", as_index=False)[["latitude", "longitude"]]
        .median()
        .rename(columns={"latitude": "lat", "longitude": "lon"})
    )


def load_wind_daily() -> pd.DataFrame:
    w = pd.read_csv(WEATHER, usecols=["date", "wspd"])
    d = pd.read_csv(WIND_DIR, usecols=["date", "wind_dir_deg"])
    w["date"] = pd.to_datetime(w["date"], errors="coerce")
    d["date"] = pd.to_datetime(d["date"], errors="coerce")
    wind = w.merge(d, on="date", how="outer").sort_values("date")
    wind["wspd"] = pd.to_numeric(wind["wspd"], errors="coerce")
    wind["wind_dir_deg"] = pd.to_numeric(wind["wind_dir_deg"], errors="coerce")
    wind["wspd"] = interp_linear(wind["wspd"])
    wind["wind_dir_deg"] = interp_linear(wind["wind_dir_deg"])
    wind = wind.dropna(subset=["date", "wspd", "wind_dir_deg"]).copy()
    return wind


def average_wind_vector(wind: pd.DataFrame, start: str, end: str) -> tuple[float, float]:
    mask = (wind["date"] >= pd.Timestamp(start)) & (wind["date"] <= pd.Timestamp(end))
    sub = wind.loc[mask].copy()
    if sub.empty:

        mask = (wind["date"] >= pd.Timestamp("2020-01-01")) & (wind["date"] <= pd.Timestamp("2025-12-31"))
        sub = wind.loc[mask].copy()


    theta_to = np.deg2rad((sub["wind_dir_deg"].to_numpy() + 180.0) % 360.0)
    v = sub["wspd"].to_numpy()
    ux = np.mean(v * np.cos(theta_to))
    uy = np.mean(v * np.sin(theta_to))
    theta_avg = np.arctan2(uy, ux)
    v_avg = float(np.mean(sub["wspd"]))
    return v_avg, theta_avg


def obs_weights_2020_2025() -> pd.DataFrame:
    df = pd.read_csv(OBS_CRASH, usecols=["Crash.Date", "Near.Intersection.Roadway", "crash_count"])
    df["date"] = pd.to_datetime(df["Crash.Date"], errors="coerce")
    df["crash_count"] = pd.to_numeric(df["crash_count"], errors="coerce").fillna(0.0)
    df = df[(df["date"] >= "2020-01-01") & (df["date"] <= "2025-12-31")].copy()
    return (
        df.groupby("Near.Intersection.Roadway", as_index=False)["crash_count"]
        .sum()
        .rename(columns={"Near.Intersection.Roadway": "intersection", "crash_count": "weight"})
    )


def obs_chronic_hotspots_2020_2025() -> set[str]:
    df = pd.read_csv(OBS_CRASH, usecols=["Crash.Date", "Near.Intersection.Roadway", "crash_count"])
    df["date"] = pd.to_datetime(df["Crash.Date"], errors="coerce")
    df["year"] = df["date"].dt.year
    df["crash_count"] = pd.to_numeric(df["crash_count"], errors="coerce").fillna(0.0)
    df = df[(df["year"] >= 2020) & (df["year"] <= 2025)].copy()

    per_year = (
        df.groupby(["year", "Near.Intersection.Roadway"], as_index=False)["crash_count"]
        .sum()
        .rename(columns={"Near.Intersection.Roadway": "intersection"})
    )

    top_sets = []
    for y, g in per_year.groupby("year"):
        q = g["crash_count"].quantile(0.95)
        top_sets.append(set(g.loc[g["crash_count"] >= q, "intersection"]))

    counts = {}
    for s in top_sets:
        for x in s:
            counts[x] = counts.get(x, 0) + 1

    return {k for k, v in counts.items() if v >= 4}


def pred_weights_2026_2028_and_hotspots() -> tuple[pd.DataFrame, set[str]]:
    agg = {}
    yearly = {}
    usecols = ["crash_date", "intersection", "pred_mean_intersection"]
    for chunk in pd.read_csv(PRED_CRASH, usecols=usecols, chunksize=1_000_000):
        chunk["date"] = pd.to_datetime(chunk["crash_date"], errors="coerce")
        chunk = chunk[(chunk["date"] >= "2026-01-01") & (chunk["date"] <= "2028-12-31")].copy()
        if chunk.empty:
            continue
        chunk["pred_mean_intersection"] = pd.to_numeric(chunk["pred_mean_intersection"], errors="coerce").fillna(0.0)
        chunk["year"] = chunk["date"].dt.year

        s = chunk.groupby("intersection")["pred_mean_intersection"].sum()
        for k, v in s.items():
            agg[k] = agg.get(k, 0.0) + float(v)

        ys = chunk.groupby(["year", "intersection"])["pred_mean_intersection"].sum()
        for (yy, inter), v in ys.items():
            yearly[(int(yy), inter)] = yearly.get((int(yy), inter), 0.0) + float(v)

    w = pd.DataFrame({"intersection": list(agg.keys()), "weight": list(agg.values())})

    # persistence for 3 years --> >=2 years in yearly 95th percentile
    top_sets = []
    yearly_df = pd.DataFrame(
        [(y, i, v) for (y, i), v in yearly.items()],
        columns=["year", "intersection", "pred_sum"]
    )
    for y, g in yearly_df.groupby("year"):
        q = g["pred_sum"].quantile(0.95)
        top_sets.append(set(g.loc[g["pred_sum"] >= q, "intersection"]))
    counts = {}
    for s in top_sets:
        for x in s:
            counts[x] = counts.get(x, 0) + 1
    hotspots = {k for k, v in counts.items() if v >= 2}
    return w, hotspots


def rotation(theta: float) -> np.ndarray:
    c, s = np.cos(theta), np.sin(theta)
    return np.array([[c, -s], [s, c]])


def sigma_matrix(v: float, theta: float) -> np.ndarray:
    stretch = max(1.0 + C_SCALE * v, 1e-6)
    lam1 = (H_DEG ** 2) * stretch
    lam2 = (H_DEG ** 2) / stretch
    R = rotation(theta)
    L = np.array([[lam1, 0.0], [0.0, lam2]])
    return R @ L @ R.T


def load_basemap_patches() -> tuple[list[Polygon], tuple[float, float, float, float]]:
    with open(BASEMAP, "r", encoding="utf-8") as f:
        gj = json.load(f)

    patches = []
    minx = miny = float("inf")
    maxx = maxy = float("-inf")

    for feat in gj["features"]:
        geom = feat.get("geometry", {})
        gtype = geom.get("type")
        coords = geom.get("coordinates", [])

        if gtype == "Polygon":
            polys = [coords]
        elif gtype == "MultiPolygon":
            polys = coords
        else:
            continue

        for poly in polys:
            if not poly:
                continue
            ring = np.array(poly[0], dtype=float)
            if ring.shape[0] < 3:
                continue
            minx = min(minx, np.nanmin(ring[:, 0]))
            maxx = max(maxx, np.nanmax(ring[:, 0]))
            miny = min(miny, np.nanmin(ring[:, 1]))
            maxy = max(maxy, np.nanmax(ring[:, 1]))
            patches.append(Polygon(ring, closed=True))

    return patches, (minx, miny, maxx, maxy)


def build_grid(bounds: tuple[float, float, float, float]) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    minx, miny, maxx, maxy = bounds
    padx = (maxx - minx) * 0.08
    pady = (maxy - miny) * 0.08
    xs = np.linspace(minx - padx, maxx + padx, GRID_N)
    ys = np.linspace(miny - pady, maxy + pady, GRID_N)
    X, Y = np.meshgrid(xs, ys)
    return X, Y, np.stack([X.ravel(), Y.ravel()], axis=1)


def anisotropic_surface(points: np.ndarray, weights: np.ndarray, grid_points: np.ndarray, Sigma: np.ndarray) -> np.ndarray:
    invS = np.linalg.inv(Sigma)
    detS = np.linalg.det(Sigma)
    norm = 1.0 / (2.0 * np.pi * np.sqrt(detS))
    z = np.zeros(grid_points.shape[0], dtype=float)


    chunk = 250
    for i in range(0, points.shape[0], chunk):
        p = points[i:i + chunk]
        w = weights[i:i + chunk]
        diff = grid_points[:, None, :] - p[None, :, :]  # G x C x 2
        q = np.einsum("gci,ij,gcj->gc", diff, invS, diff)
        z += np.sum(w[None, :] * norm * np.exp(-0.5 * q), axis=1)

    return z


def add_compass(ax, theta_rad: float, title: str = "Prevailing wind"):

    x0, y0 = 0.12, 0.14
    r = 0.06
    ax.add_patch(plt.Circle((x0, y0), r, transform=ax.transAxes, fill=False, linewidth=1.0, color="black"))
    dx = r * np.cos(theta_rad)
    dy = r * np.sin(theta_rad)
    ax.annotate(
        "",
        xy=(x0 + dx, y0 + dy),
        xytext=(x0, y0),
        xycoords=ax.transAxes,
        arrowprops=dict(arrowstyle="-|>", lw=1.5, color="black"),
    )
    ax.text(x0, y0 + r + 0.025, title, transform=ax.transAxes, ha="center", va="bottom", fontsize=8)
    ax.text(x0, y0 - r - 0.02, "plume direction", transform=ax.transAxes, ha="center", va="top", fontsize=7)


def make_map(weights_df: pd.DataFrame, start: str, end: str, out_path: Path, title: str):
    coords = load_coords()
    w = weights_df.merge(coords, on="intersection", how="inner")
    w = w[w["weight"] > 0].copy()
    if w.empty:
        raise RuntimeError(f"No weighted intersections for {start} to {end}")

    # Normalize weights
    w["weight"] = w["weight"] / (w["weight"].max() + EPS)

    wind = load_wind_daily()
    v_avg, theta_avg = average_wind_vector(wind, start, end)
    Sigma = sigma_matrix(v_avg, theta_avg)

    patches, bounds = load_basemap_patches()
    X, Y, GP = build_grid(bounds)

    pts = w[["lon", "lat"]].to_numpy()
    wt = w["weight"].to_numpy()
    Z = anisotropic_surface(pts, wt, GP, Sigma).reshape(X.shape)
    Z = Z / (np.nanmax(Z) + EPS)

    q95 = np.nanquantile(Z, 0.95)

    fig, ax = plt.subplots(figsize=(10, 10), dpi=220)
    for p in patches:
        p.set_facecolor("#f6f6f6")
        p.set_edgecolor("#999999")
        p.set_linewidth(0.4)
        ax.add_patch(p)


    alpha = np.clip(Z ** 0.65, 0, 1)
    hm = ax.imshow(
        Z,
        extent=[X.min(), X.max(), Y.min(), Y.max()],
        origin="lower",
        cmap="YlOrRd",
        alpha=alpha * 0.95,
        zorder=2,
    )

    # 95th percentile hotspot
    ax.contour(
        X, Y, Z,
        levels=[q95],
        colors=["#7f0000"],
        linewidths=1.3,
        zorder=3
    )

    add_compass(ax, theta_avg, title="Avg wind")
    cbar = fig.colorbar(hm, ax=ax, fraction=0.028, pad=0.02)
    cbar.set_label("Relative anisotropic KDE intensity", fontsize=9)

    ax.set_title(title, fontsize=13, pad=8)
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlim(X.min(), X.max())
    ax.set_ylim(Y.min(), Y.max())
    fig.tight_layout()
    fig.savefig(out_path, dpi=220)
    plt.close(fig)


def main():
    obs_w = obs_weights_2020_2025()
    make_map(
        weights_df=obs_w,
        start="2020-01-01",
        end="2025-12-31",
        out_path=OUT_2020_2025,
        title="Aggregate Anisotropic Heatmap (2020-2025)",
    )

    pred_w, _pred_hot = pred_weights_2026_2028_and_hotspots()
    make_map(
        weights_df=pred_w,
        start="2026-01-01",
        end="2028-12-31",
        out_path=OUT_2026_2028,
        title="Aggregate Anisotropic Heatmap (2026-2028, NB Forecast)",
    )

    print("WROTE", OUT_2020_2025)
    print("WROTE", OUT_2026_2028)


if __name__ == "__main__":
    main()
